In [ ]:
import numpy as np
import pandas as pd

train_df=pd.read_csv('datasets/playground-series-s5e6/train.csv')
test_df=pd.read_csv('datasets/playground-series-s5e6/test.csv')

In [ ]:
# train_df.info()
# train_df.head()
# train_df.describe()
# train_df.isna().sum()

In [ ]:
def preprocess_data(df):
    df=pd.get_dummies(df, columns=['Soil Type', 'Crop Type'], drop_first=True)
    return df
    
train_df=preprocess_data(train_df)
test_df=preprocess_data(test_df)
train_df=train_df.drop(columns=['id'])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cols=['Temparature', 'Humidity', 'Moisture', 'Nitrogen', 'Potassium', 'Phosphorous']
# for col in cols:
#     sns.histplot(train_df[col], kde=True)
#     plt.title(f'Histplot - {col}')
#     plt.show()

#     sns.boxplot(train_df[col])
#     plt.title(f'Boxplot - {col}')
#     plt.show()

In [ ]:
for col in cols:
    Q1=train_df[col].quantile(0.25)
    Q3=train_df[col].quantile(0.75)
    IQR=Q3-Q1

    lower_bound=Q1-1.5*IQR
    upper_bound=Q3+1.5*IQR

    # print(f'Lower Bound[{col}]: {lower_bound}')
    # print(f'Upper Bound[{col}]: {upper_bound}')

In [ ]:
from scipy.stats import skew
for col in cols:
    skewness=skew(train_df[col])
    # print(f'Skewness[{col}]: {skewness}')

In [ ]:
# corr=train_df.corr(numeric_only=True)
# sns.heatmap(corr, annot=True, cmap='coolwarm', center=0)
# plt.title('Feature Correlation Heatmap')
# plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV

from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

X=train_df.drop(columns=['Fertilizer Name'])
y=train_df['Fertilizer Name']

X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, random_state=42)

param_grid = {
    'tree__max_depth': [None, 3, 5, 10, 15, 20],
    'tree__min_samples_split': [2, 5, 10, 20],
    'tree__min_samples_leaf': [1, 2, 5, 10],
    'tree__max_features': [None, 'sqrt', 'log2'],
    'tree__criterion': ['gini', 'entropy', 'log_loss'],
    'tree__splitter': ['best', 'random'],
    'tree__ccp_alpha': [0.0, 0.01, 0.05, 0.1]
}

pipeline=Pipeline([
    ('tree', DecisionTreeClassifier(random_state=42))
])

grid_search=GridSearchCV(pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=-1)

grid_search.fit(X_train, y_train)

predictions=grid_search.predict(X_test)

accuracy=accuracy_score(y_test, predictions)
print("Done")

In [ ]:
# 1. Accuracy - Bar Plot
print("[ ACCURACY ]")
plt.figure(figsize=(4, 4))
plt.bar(["Accuracy"], [accuracy], color='skyblue')
plt.ylim(0, 1)
plt.title("Model Accuracy")
plt.ylabel("Score")
plt.show()

# 2. Actual vs Predicted - Bar Chart
results_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': predictions
})

plt.figure(figsize=(8, 4))
sns.countplot(
    x='value', hue='Type',
    data=pd.melt(results_df, value_vars=['Actual', 'Predicted'], var_name='Type'),
    palette='pastel'
)
plt.title("Actual vs Predicted Class Counts")
plt.xlabel("Class")
plt.ylabel("Count")
plt.legend(title='Label Type')
plt.show()

# 3. Confusion Matrix - Heatmap
cm = confusion_matrix(y_test, predictions)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# 4. Precision, Recall, F1 Score - Heatmap
report = classification_report(y_test, predictions, output_dict=True, zero_division=0)

# Convert to DataFrame (remove 'accuracy' to keep it 2D for heatmap)
report_df = pd.DataFrame(report).drop(columns=['accuracy'])
plt.figure(figsize=(8, 5))
sns.heatmap(report_df.iloc[:3, :].T, annot=True, cmap="YlGnBu", fmt=".2f")
plt.title("Classification Report")
plt.ylabel("Class")
plt.xlabel("Metrics")
plt.tight_layout()
plt.show()

# 5. Cross-Validation Scores - Bar Plot
scores = cross_val_score(grid_search, X, y, cv=3)
plt.figure(figsize=(8, 4))
plt.bar(range(1, 4), scores, color='lightgreen')
plt.ylim(0, 1)
plt.title("Cross-Validation Accuracy per Fold")
plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.xticks(range(1, 4))
plt.show()

In [ ]:
import joblib

X_test_final = test_df.drop(columns=['Fertilizer Name'], errors='ignore')

test_predictions = grid_search.best_estimator_.predict(X_test_final)

submission = pd.DataFrame({
    'Fertilizer Name': test_predictions
})

submission.to_csv('submission.csv', index=False)
joblib.dump(grid_search.best_estimator_, 'OptimalFertilizers_DecisionTree.joblib')